In [ ]:
%cd /home/anw2067/visualnav-transformer/train
import argparse
from datetime import datetime
import os
import torch
import yaml
import copy
import wandb
import json
import random


from nymeria.download_utils import DownloadManager
from nymeria.definitions import DataGroups
from nymeria.data_provider import SequencePathProvider, NymeriaDataProvider
from nymeria.definitions import Subpaths, VrsFiles
from nymeria.recording_data_provider import create_recording_data_provider

import numpy as np
from torchvision import transforms
from dreamsim import dreamsim
from scipy.spatial.transform import Rotation as R
from torch.utils.data import DistributedSampler, RandomSampler, DataLoader
from diffusers.models import AutoencoderKL

from peva.models import CDiT_models
from peva.diffusion import create_diffusion

from vint_train.training.nymeria_training_utils import get_action_smpl_torch
from vint_train.data.misc import XSensConstants, XsensSkeleton
from planning.utils import _compute_pose_and_loss, _compute_part_distance_matrices
from planning.cem import CEMPlanner
from planning.utils import get_nymeria_dataset, load_peva, load_policy
from planning.wrappers import EvaluatorPeva, EvaluatorWaypoint, ObjectiveDreamSIM, PevaWM, Preprocessor, WaypointWM
from planning.sampling import waypoint_sample
from planning.vis_utils import *
from planning.sampling import policy_sample

from torchvision.utils import save_image

# OUTPUT_DIR = "/home/anw2067/visualnav-transformer/train/logs/paper_vis/fig3"
DATA_SAVE_DIR = "/home/anw2067/scratch/nymeria_camera_dir"
DATA_JSON="/home/anw2067/visualnav-transformer/data_jsons/visibility_no_data.json"

os.makedirs(DATA_SAVE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


/home/anw2067/visualnav-transformer/train


/scratch/anw2067/conda/envs/nomad_train2/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]
/scratch/anw2067/conda/envs/nomad_train2/lib/python3.10/site-packages/wandb/sdk/internal/internal_api.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/scratch/anw2067/conda/envs/nomad_train2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
GLOBAL_POLICY = None
GLOBAL_POLICY_DIFFUSION = None
GLOBAL_NOMAD_STATS = None
GLOBAL_NOMAD_CONFIG = None
GLOBAL_PEVA_MODEL = None
GLOBAL_PEVA_DIFFUSION = None
GLOBAL_PEVA_VAE = None
GLOBAL_PEVA_STATS = None
GLOBAL_PEVA_CONFIG = None


In [6]:
from torchvision.utils import draw_keypoints


def draw_waypoints_vis(obs, waypoints, color_order=["red", "green", "blue", "yellow"], radius=4):
    """
    Draws waypoint as circles on an image
    
    Args:
        obs: B, 3, H, W 
        waypoints: B, 8 or B, 4, 2
        color_order: list of colors
    """
    B = obs.shape[0]
    
    device = obs.device
    output_images = []
    if waypoints.shape[-1] == 8:
        waypoints = waypoints.reshape(B, 4, 2)
    for b in range(B):
        image = torch.clone(obs[b]) 
        for index, color in enumerate(color_order):
            if (waypoints[b, index:index+1] == -1).all(): continue
            image = draw_keypoints(image, waypoints[b, index:index+1, None, :], colors="black", radius=radius+1)
            image = draw_keypoints(image, waypoints[b, index:index+1, None, :], colors=color, radius=radius)
        output_images.append(image)
    return torch.stack(output_images, dim=0).to(device)

In [8]:
import importlib
import planning.vis_utils
importlib.reload(planning.utils)
from planning.utils import draw_waypoints
importlib.reload(planning.vis_utils)
from planning.vis_utils import *
import vint_train.data.misc
importlib.reload(vint_train.data.misc)
from vint_train.data.misc import XSensConstants, XsensSkeleton 
from pathlib import Path
disable_logging()

def main(args):
    seed = 42
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    OUTPUT_DIR = f"/home/anw2067/visualnav-transformer/train/logs/paper_vis/fig3-{seed}"
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    global GLOBAL_POLICY, GLOBAL_POLICY_DIFFUSION, GLOBAL_NOMAD_STATS, GLOBAL_NOMAD_CONFIG
    if GLOBAL_POLICY is None:
        GLOBAL_POLICY, GLOBAL_POLICY_DIFFUSION, GLOBAL_NOMAD_STATS, GLOBAL_NOMAD_CONFIG = load_policy(args.nomad_config, args.nomad_checkpoint, device=device)
    policy, policy_diffusion, nomad_stats, nomad_config = GLOBAL_POLICY, GLOBAL_POLICY_DIFFUSION, GLOBAL_NOMAD_STATS, GLOBAL_NOMAD_CONFIG
    # policy, policy_diffusion, nomad_stats, nomad_config = load_policy(args.nomad_config, args.nomad_checkpoint, device=device)
    # peva_model, _, peva_diffusion, peva_vae, peva_stats, peva_config = load_peva(args.peva_config, args.peva_checkpoint, device=device,
                                                                    # inference_context_size=args.peva_context_size,
                                                                    # diffusion_steps=args.peva_diffusion_steps)
    
    dataset = get_nymeria_dataset(nomad_config, context_size=max(args.peva_context_size-1, nomad_config["context_size"]), goal_timestep_offset=args.goal_timestep_offset)
    sampler = DistributedSampler(dataset, num_replicas=1, rank=0, shuffle=args.shuffle, seed=seed)
    dataloader = DataLoader(dataset, batch_size=1, sampler=sampler, num_workers=1)
    
    waypoints_bank = {1: [], 2: [], 3: [], 4: []}
    for idx, batch in enumerate(dataloader):
        if idx > 50: break
        waypoints = batch["goal_image_coords"].cuda()[:, XSensConstants.leaf_indices]
        num_visible = (waypoints != -1).all(dim=-1).sum()
        if num_visible == 0:
            continue
        waypoints_bank[num_visible.item()].append(waypoints)
    
    prev_track_name = None
    for idx, batch in enumerate(dataloader):
        if idx > 50: break
        obs_images = batch["obs_images"] # 1, context_size, 3, H, W
        goal_image = batch["goal_image"] # 1, 3, H, W
        context_poses = batch["context_poses"] # 1, context_size, 48

        deltas = batch["deltas"] # 1, horizon, action_dim
        first_pose = batch["first_pose"] # 1, 1, 48
        xsens_offsets = batch["xsens_offsets"][0] # 1, 15, 3
        goal_obs = batch["goal_obs"] # 1, 3, H, W
        goal_image_coords = batch["goal_image_coords"] # 1, 23, 2
        
        dataset_index = batch["dataset_index"].item()
        track_name, track_index = batch["dataset_track"][0], batch["dataset_track_index"].item()
        track_idx_name = f"{track_name}-{track_index}"
        
        skel = XsensSkeleton(xsens_offsets)
        gt_actions = get_action_smpl_torch(first_pose, deltas, XSensConstants.upper_body_num_parts) # B, T, 48
        xyz_dist_matrix, _, init_xyz, _ = _compute_part_distance_matrices(first_pose[:, -1], gt_actions[:, -1], skel)
        visible_plus_head = (goal_image_coords != -1).all(dim=-1)[:, :XSensConstants.upper_body_num_parts] # B, num_parts
        visible_plus_head[:, XSensConstants.part_names.index("Head")] = True
        init_visible_plus_head = xyz_dist_matrix[:, XSensConstants.leaf_indices] * visible_plus_head[:, XSensConstants.leaf_indices]
        init_visible_plus_head = (init_visible_plus_head.sum() / visible_plus_head.sum()).item()
        
        if not args.keep_nonvisible_goal:
            visible = False
            find_count = 0
            for part in ["Pelvis", "Head", "R_Hand", "L_Hand"]:
                index = XSensConstants.part_names.index(part)
                if all(goal_image_coords[0, index] != -1):
                    find_count += 1
                    if find_count > 0:
                        visible = True
                        break
            if not visible:
                print(f"No visible parts in {track_idx_name}")
                continue
            
        if init_visible_plus_head < args.min_dist_threshold:
            print(f"Initial distance of visible + head joints is less than {args.min_dist_threshold} in {track_idx_name}")
            continue
        print(f"Visualizing {track_idx_name}")
        
        curr_save_dir = f"{OUTPUT_DIR}/{track_idx_name}"
        os.makedirs(curr_save_dir, exist_ok=True)
        save_img = torch.cat([obs_images, goal_obs[None],torch.zeros_like(obs_images[:, :-2]), goal_image[None]], dim=1)[0]
        save_image(save_img, f"{curr_save_dir}/context_and_goal.png", nrow=obs_images.shape[1])
        
        # Duplicate all batches by 64
        batch_size_dup = args.num_batch_repeats
        
        obs_images = obs_images.to(device)
        context_poses = context_poses.to(device)
        goal_obs = goal_obs.to(device)
        goal_image = goal_image.to(device)
        goal_image_coords = goal_image_coords.to(device)
        first_pose = first_pose.to(device)
        gt_actions = gt_actions.to(device)
        deltas = deltas.to(device)
        
        # config details
        policy_pred_horizon = nomad_config["len_traj_pred"]
        policy_action_dim = nomad_config["input_dims"]
        policy_context_size = nomad_config["context_size"] + 1
        
        policy_context_poses = context_poses[:, -policy_context_size:]
        
        for num_visible in waypoints_bank:
            for wp_idx, wp in enumerate(waypoints_bank[num_visible]):
                counterfactual_goal = draw_waypoints(obs_images[:, -1], wp)
                pred_delta = policy_sample(policy, policy_diffusion,
                                            obs_images[:, -policy_context_size:],
                                            counterfactual_goal,
                                            policy_context_poses,
                                            policy_pred_horizon,
                                            policy_action_dim,
                                            device,)
                pred_actions = get_action_smpl_torch(first_pose, pred_delta, XSensConstants.upper_body_num_parts)
                gt_actions = get_action_smpl_torch(first_pose, deltas, XSensConstants.upper_body_num_parts)
                
                if not os.path.exists(os.path.join(DATA_SAVE_DIR, track_name)):
                    os.makedirs(os.path.join(DATA_SAVE_DIR, track_name))
                    print(f"downloading episode {track_name}")
                    download_episode(DATA_JSON, DATA_SAVE_DIR, track_name)
                
                if track_name != prev_track_name:
                    nymeria_dp = NymeriaDataProvider(sequence_rootdir=Path(os.path.join(DATA_SAVE_DIR, track_name)), load_wrist=False, load_observer=False)
                    cam_model = load_camera_model(DATA_SAVE_DIR, track_name)
                prev_track_name = track_name
                
                T_C_Pelvis = get_T_C_pelvis(nymeria_dp, track_index)
                
                goal_image_large = draw_waypoints_vis(obs_images[:, -1], goal_image_coords[:, XSensConstants.leaf_indices], radius=6)
                gt_drawn = [goal_image_large[0].cpu()]
                for t in range(gt_actions.shape[1]):
                    image = transforms.ToPILImage()(goal_image[0])
                    draw = ImageDraw.Draw(image)
                    gc_image_coords = pose_to_image_coords(gt_actions[:, t], cam_model, xsens_offsets, T_C_Pelvis) # B, 15, 2
                    gc_vis = draw_image_coords(draw, gc_image_coords, color=(int(255), int(255), int(255)), show_text=True)
                    gt_drawn.append(transforms.ToTensor()(image))
                    
                gt_stack = torch.stack(gt_drawn, dim=0)
                save_image(gt_stack, f"{OUTPUT_DIR}/{track_idx_name}.png", nrow=gt_stack.shape[0])
                
                large_counterfactual_goal = draw_waypoints_vis(obs_images[:, -1], wp, radius=6)
                pred_drawn = [large_counterfactual_goal[0].cpu()]
                for t in range(pred_actions.shape[1]):
                    image = transforms.ToPILImage()(counterfactual_goal[0])
                    draw = ImageDraw.Draw(image)
                    gc_image_coords = pose_to_image_coords(pred_actions[:, t], cam_model, xsens_offsets, T_C_Pelvis) # B, 15, 2
                    gc_vis = draw_image_coords(draw, gc_image_coords, color=(int(255), int(255), int(255)), show_text=True)
                    pred_drawn.append(transforms.ToTensor()(image))
                
                stack = torch.stack([*pred_drawn], dim=0)
                save_image(stack, f"{curr_save_dir}/{num_visible}-{wp_idx}-stacked_actions.png", nrow=len(gt_drawn))
        
                
        

MODEL_DIRECTORY={
    "draw": (
        "/home/anw2067/visualnav-transformer/train/logs/nomad-minimal/2025_12_09_11_24:nomad-minimal-proprioception-cat8-dinov3_unpool_3dposemb-proj-lr5e-4-pool_curr_obs-goaldraw/config.yaml",
        "/home/anw2067/visualnav-transformer/train/logs/nomad-minimal/2025_12_09_11_24:nomad-minimal-proprioception-cat8-dinov3_unpool_3dposemb-proj-lr5e-4-pool_curr_obs-goaldraw/ema_9.pth"
    ),
    "gravity": (
        "/home/anw2067/visualnav-transformer/train/logs/nomad-minimal/2025_12_18_11_47:nomad-minimal-proprioception-cat8-dinov3_unpool_3dposemb-proj-lr5e-4-pool_curr_obs-goaldraw-preserveUpDown/config.yaml",
        "/home/anw2067/visualnav-transformer/train/logs/nomad-minimal/2025_12_18_11_47:nomad-minimal-proprioception-cat8-dinov3_unpool_3dposemb-proj-lr5e-4-pool_curr_obs-goaldraw-preserveUpDown/ema_9.pth"
    ),
    "draw_mask": (
        "/home/anw2067/visualnav-transformer/train/config/torch/minimal-nomad-proprioception-cat8-dinov3_unpool_3dposemb-proj-lr5e-4-pool_curr_obs-goaldraw-waypointMask.yaml",
        "/home/anw2067/visualnav-transformer/train/logs/nomad-minimal/2026_01_21_06_54:nomad-minimal-proprioception-cat8-dinov3_unpool_3dposemb-proj-lr5e-4-pool_curr_obs-goaldraw-waypointMask/ema_9.pth"
    )
}
        
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    
    parser.add_argument("-a", "--algo", type=str, choices=["peva", "waypoint"], default="waypoint", help="Planning algorithm")
    parser.add_argument("--use_leafxyz_as_cost", action='store_true', help="Uses the metric(leaf-xyz) instead of a normal cost_fn")
    parser.add_argument("--goal_timestep_offset", type=int, default=None, help="Goal timestep offset")
    parser.add_argument("--shuffle", action="store_true", help="Shuffle the dataset")
    
    parser.add_argument("--num_batch_repeats", type=int, default=8, help="Number of batch repeats")
    parser.add_argument("--keep_nonvisible_goal", action="store_true", help="Keep non-visible goal in the dataset")
    parser.add_argument("--min_index_goal", type=int, default=0, help="Minimum index of the goal to plan")
    parser.add_argument("--min_dist_threshold", type=float, default=0.1, help="Minimum distance threshold")
    parser.add_argument("--num_samples_to_plan", type=int, default=32, help="Number of samples to plan")
    parser.add_argument("--no_wandb", action="store_true", help="Don't use wandb")
    parser.add_argument("--test", action="store_true", help="Test run")
    
    parser.add_argument("--peva_config", type=str, default="/home/anw2067/visualnav-transformer/train/peva/config/nymeria_rel_concat_embedding_compile_beta095_ar_model_context_16_bs_16_smpl_lowebody_-64to_64_1_goal_emb_relative_xxl.yaml")
    parser.add_argument("--peva_checkpoint", type=str, default="/scratch/anw2067/nymeria_rel_concat_embedding_compile_beta095_ar_model_context_16_bs_16_smpl_lowebody_cancel_scaler_-64to_64_xxl_280_0180000.pth.tar")
    parser.add_argument("--peva_context_size", type=int, default=15, help="PEVA context size")
    parser.add_argument("--peva_diffusion_steps", type=int, default=250, help="PEVA diffusion steps")
    
    parser.add_argument("--nomad_model", type=str, default="draw", choices=["draw", "gravity", "draw_mask"])
    parser.add_argument("--nomad_config", type=str, default=None)
    parser.add_argument("--nomad_checkpoint", type=str, default=None)
    
    parser.add_argument("--world_size", type=int, default=1, help="World size")
    parser.add_argument("--rank", type=int, default=0, help="Rank")
    
    # In Jupyter notebooks, pass arguments as a list to parse_args() instead of using sys.argv
    # Pass an empty list [] to use all defaults, or specify arguments like: ['--shuffle', '--nomad_model', 'draw']
    args = parser.parse_args(["--shuffle", "--nomad_model", "draw_mask", "--peva_context_size", "7"])
    
    if args.nomad_model is not None:
        assert args.nomad_config is None and args.nomad_checkpoint is None
        args.nomad_config, args.nomad_checkpoint = MODEL_DIRECTORY[args.nomad_model]
    else:
        assert args.nomad_config is not None and args.nomad_checkpoint is not None
    
    main(args)

Visualizing 20231109_s0_janet_walsh_act3_1uklre-1131


[ProgressLogger][INFO]: 2026-01-28 04:43:06: Opening /home/anw2067/scratch/nymeria_camera_dir/20231109_s0_janet_walsh_act3_1uklre/recording_head/data/motion.vrs...
[MultiRecordFileReader][DEBUG]: Opened file '/home/anw2067/scratch/nymeria_camera_dir/20231109_s0_janet_walsh_act3_1uklre/recording_head/data/motion.vrs' and assigned to reader #0
[VrsDataProvider][INFO]: streamId 247-1/baro0 activated
[VrsDataProvider][WARNING]: Unsupported TimeSync mode: APP, ignoring.
[VrsDataProvider][INFO]: Timecode stream found: 285-2
[VrsDataProvider][INFO]: streamId 1202-1/imu-right activated
[VrsDataProvider][INFO]: streamId 1202-2/imu-left activated
[VrsDataProvider][INFO]: streamId 1203-1/mag0 activated
[MpsDataPathsProvider][WARNING]: Hand tracking folder (/home/anw2067/scratch/nymeria_camera_dir/20231109_s0_janet_walsh_act3_1uklre/recording_head/mps/hand_tracking) does not exist in MPS root folder, not loading wrist and palm poses.
[MpsDataPathsProvider][WARNING]: Hand tracking folder (/home/anw

Loaded #closed loop trajectory poses records: 1129414


[ProgressLogger][INFO]: 2026-01-28 04:43:15: Opening /home/anw2067/scratch/nymeria_camera_dir/20231109_s0_janet_walsh_act3_1uklre/recording_head/data/motion.vrs...
[MultiRecordFileReader][DEBUG]: Opened file '/home/anw2067/scratch/nymeria_camera_dir/20231109_s0_janet_walsh_act3_1uklre/recording_head/data/motion.vrs' and assigned to reader #0
[VrsDataProvider][INFO]: streamId 247-1/baro0 activated
[VrsDataProvider][WARNING]: Unsupported TimeSync mode: APP, ignoring.
[VrsDataProvider][INFO]: Timecode stream found: 285-2
[VrsDataProvider][INFO]: streamId 1202-1/imu-right activated
[VrsDataProvider][INFO]: streamId 1202-2/imu-left activated
[VrsDataProvider][INFO]: streamId 1203-1/mag0 activated


Visualizing 20231110_s0_thomas_brown_act3_pisdac-1387


...

No visible parts in 20231206_s1_virginia_perez_act1_smf6ci-2627
No visible parts in 20231220_s0_victor_sloan_act4_v05i2p-1349
No visible parts in 20230712_s0_laura_wilson_act0_yg1c18-2303
Initial distance of visible + head joints is less than 0.1 in 20231127_s1_robyn_blackburn_act3_t4abeq-458
Initial distance of visible + head joints is less than 0.1 in 20230817_s0_brittney_powell_act1_tdosac-584
Initial distance of visible + head joints is less than 0.1 in 20231208_s0_ronald_guerra_act5_y2gscm-1045
Initial distance of visible + head joints is less than 0.1 in 20231110_s0_thomas_brown_act3_pisdac-949
No visible parts in 20231113_s1_greg_clark_act2_jc6wnc-4213
Visualizing 20231219_s0_randall_love_act6_8pqkk5-754


[ProgressLogger][INFO]: 2026-01-28 04:48:15: Opening /home/anw2067/scratch/nymeria_camera_dir/20231219_s0_randall_love_act6_8pqkk5/recording_head/data/motion.vrs...
[MultiRecordFileReader][DEBUG]: Opened file '/home/anw2067/scratch/nymeria_camera_dir/20231219_s0_randall_love_act6_8pqkk5/recording_head/data/motion.vrs' and assigned to reader #0
[VrsDataProvider][INFO]: streamId 247-1/baro0 activated
[VrsDataProvider][WARNING]: Unsupported TimeSync mode: APP, ignoring.
[VrsDataProvider][INFO]: Timecode stream found: 285-2
[VrsDataProvider][INFO]: streamId 1202-1/imu-right activated
[VrsDataProvider][INFO]: streamId 1202-2/imu-left activated
[VrsDataProvider][INFO]: streamId 1203-1/mag0 activated
[MpsDataPathsProvider][WARNING]: Hand tracking folder (/home/anw2067/scratch/nymeria_camera_dir/20231219_s0_randall_love_act6_8pqkk5/recording_head/mps/hand_tracking) does not exist in MPS root folder, not loading wrist and palm poses.
[MpsDataPathsProvider][WARNING]: Hand tracking folder (/home/

Loaded #closed loop trajectory poses records: 1035552


[ProgressLogger][INFO]: 2026-01-28 04:48:23: Opening /home/anw2067/scratch/nymeria_camera_dir/20231219_s0_randall_love_act6_8pqkk5/recording_head/data/motion.vrs...
[MultiRecordFileReader][DEBUG]: Opened file '/home/anw2067/scratch/nymeria_camera_dir/20231219_s0_randall_love_act6_8pqkk5/recording_head/data/motion.vrs' and assigned to reader #0
[VrsDataProvider][INFO]: streamId 247-1/baro0 activated
[VrsDataProvider][WARNING]: Unsupported TimeSync mode: APP, ignoring.
[VrsDataProvider][INFO]: Timecode stream found: 285-2
[VrsDataProvider][INFO]: streamId 1202-1/imu-right activated
[VrsDataProvider][INFO]: streamId 1202-2/imu-left activated
[VrsDataProvider][INFO]: streamId 1203-1/mag0 activated


Initial distance of visible + head joints is less than 0.1 in 20231220_s0_victor_sloan_act0_0afk6m-3409
Initial distance of visible + head joints is less than 0.1 in 20230829_s0_ray_humphrey_act4_7lkmhe-1493
Initial distance of visible + head joints is less than 0.1 in 20231115_s1_andrew_johnson_act6_vol5wd-2529
